# Hafta 10 - Görev 2: GPT-2'yi Kendin Yaz ve Ağırlıkları Yükle

**Görev Tanımı:**
> GPT-2'yi kendin yaz ve ağırlıkları yükle. Hafta 8'deki GPT'nden yola çıkarak GPT-2 modülünü yaz. Hugging Face ağırlıklarını kendi modeline yükle. Aynı girdi için kendi modelinin logits'inin Hugging Face modelininkiyle aynı çıktığını göster. Hafta 8'deki GPT'nden farklarını madde madde yaz, örneğin aktivasyon fonksiyonu ve q, k, v'nin tek katmanda hesaplanması. Bazı ağırlıkları yüklerken neden transpoze etmen gerektiğini de yaz. (0:13:47 - 0:31:00)


### 1. GPT-2 (124M) Konfigürasyonu
Karpathy'nin *build-nanogpt* reposundaki gibi GPT-2 taban modeli (124M) için konfigürasyon sınıfını oluşturuyoruz:
- `block_size = 1024`: Maksimum bağlam penceresi
- `vocab_size = 50257`: BPE sözlük boyutu
- `n_layer = 12`: Transformer blok sayısı
- `n_head = 12`: Çoklu dikkat kafa sayısı
- `n_embd = 768`: Gömme boyutu (her kafa 768 / 12 = 64 boyutludur)


In [1]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2LMHeadModel
import tiktoken

@dataclass
class GPTConfig:
    block_size: int = 1024   # maksimum bağlam uzunluğu
    vocab_size: int = 50257  # sözlük boyutu (50,000 BPE + 256 byte tokens + 1 özel token)
    n_layer: int = 12        # katman sayısı
    n_head: int = 12         # kafa sayısı
    n_embd: int = 768        # gömme boyutu (embedding dimension)

print("GPTConfig sınıfı hazır.")


GPTConfig sınıfı hazır.


### 2. Alt Modüllerin Tanımlanması
1. **`CausalSelfAttention`**: Q, K, V tek bir `nn.Linear(n_embd, 3 * n_embd)` katmanında birleşik olarak hesaplanır. Ardından kafalara bölünür ve nedensel (causal) alt üçgen maske ile dikkat ağırlıkları çarpılır.
2. **`MLP`**: 4 kat genişleyen `c_fc` lineer katmanı, ardından `GELU(approximate='tanh')` aktivasyonu ve `c_proj` projeksiyon katmanı.
3. **`Block`**: Pre-LayerNorm mimarisi (`ln_1 -> attn -> residual` ve `ln_2 -> mlp -> residual`).


In [2]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Q, K, V projeksiyonlarının hepsi TEK bir lineer katmanda birleşik hesaplanır
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Çıkış projeksiyon katmanı
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        
        # Nedensel maske (Causal mask): gelecekteki tokenlara bakmayı engeller
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch, Sequence Length, Embedding Dim
        
        # Q, K, V hesaplama ve ayırma
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        
        # Kafalara bölme: (B, T, n_head, head_size) -> (B, n_head, T, head_size)
        head_size = C // self.n_head
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)
        
        # Scaled dot-product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_size))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        
        # Ağırlıklı değerler toplamı
        y = att @ v # (B, n_head, T, head_size)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # kafaları tekrar birleştir
        
        # Çıkış projeksiyonu
        y = self.c_proj(y)
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        # Pre-LayerNorm mimarisi: normalizasyon residual bağlantı öncesinde uygulanır
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

print("CausalSelfAttention, MLP ve Block modülleri tanımlandı.")


CausalSelfAttention, MLP ve Block modülleri tanımlandı.


### 3. Ana GPT Modeli ve Hugging Face Ağırlık Yükleme (`from_pretrained`)
Modelimizi oluşturuyoruz. `from_pretrained` metodu ile:
- Hugging Face'teki `gpt2` modelinin ağırlıklarını alıyoruz.
- OpenAI checkpoint'i `Conv1D` kullandığı için PyTorch `nn.Linear` ile uyum sağlaması amacıyla ilgili 4 katmanın ağırlıklarını **transpoze** (`.t()`) ediyoruz.
- Weight tying (`wte.weight = lm_head.weight`) uyguluyoruz.


In [3]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight Tying (Ağırlık Paylaşımı)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"Girdi uzunluğu {T}, maksimum bağlam {self.config.block_size}'den büyük olamaz."

        # Pozisyon ve token embedding toplamı
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos) # (T, n_embd)
        tok_emb = self.transformer.wte(idx) # (B, T, n_embd)
        x = tok_emb + pos_emb

        # Transformer bloklarından geçiş
        for block in self.transformer.h:
            x = block(x)

        # Final LayerNorm ve Logit üretimi
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type='gpt2'):
        """Hugging Face GPT-2 ağırlıklarını sıfırdan yazdığımız modelimize yükler"""
        print(f"Hugging Face '{model_type}' modeli yükleniyor...")
        config = GPTConfig()
        model = cls(config)
        sd = model.state_dict()
        sd_keys = [k for k in sd.keys() if not k.endswith('.attn.bias')]

        # Hugging Face modelini al
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = [k for k in sd_hf.keys() if not k.endswith('.attn.masked_bias') and not k.endswith('.attn.bias')]

        # OpenAI Conv1D katmanları: PyTorch Linear katmanına yüklerken TRANSPOZE edilmelidir!
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

        assert len(sd_keys_hf) == len(sd_keys), f"Anahtar sayısı uyuşmuyor: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # Conv1D ağırlıkları ters boyuttadır (in, out), transpoze edilmelidir (out, in)
                assert sd_hf[k].shape[::-1] == sd[k].shape, f"Transpoze şekil uyuşmazlığı: {k}"
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # Standart ağırlık kopyalama
                assert sd_hf[k].shape == sd[k].shape, f"Şekil uyuşmazlığı: {k}"
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        print("Tüm ağırlıklar başarıyla eşlendi ve kopyalandı!")
        return model, model_hf

print("GPT sınıfı hazır.")


GPT sınıfı hazır.


### 4. Ağırlıkların Yüklenmesi ve Logits Karşılaştırması
Şimdi kendi modelimize ağırlıkları aktaracağız ve aynı girdi için Hugging Face modeliyle bizim modelimizin birebir aynı logits çıktısını verdiğini sayısal olarak göstereceğiz.

In [4]:
# Modelleri oluştur ve ağırlıkları yükle
model_custom, model_hf = GPT.from_pretrained('gpt2')

model_custom.eval()
model_hf.eval()

# Cihaz seçimi (GPU varsa GPU, yoksa CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_custom.to(device)
model_hf.to(device)

# Test girdisi: Karpathy'nin videoda kullandığı klasik örnek cümle
enc = tiktoken.get_encoding('gpt2')
prompt = "Hello, I'm a language model,"
tokens = enc.encode(prompt)
x = torch.tensor([tokens], dtype=torch.long, device=device)

print(f"Test Girdisi: '{prompt}'")
print(f"Girdi Token ID'leri: {tokens}")
print(f"Girdi Tensör Boyutu: {x.shape}")

# İki modelin ileri geçiş (forward pass) çıktıları
with torch.no_grad():
    our_logits, _ = model_custom(x)
    hf_logits = model_hf(x).logits

print(f"Bizim Model Logits Boyutu : {our_logits.shape}")
print(f"HF Model Logits Boyutu    : {hf_logits.shape}")

# Sayısal fark analizi
max_abs_diff = torch.max(torch.abs(our_logits - hf_logits)).item()
mean_abs_diff = torch.mean(torch.abs(our_logits - hf_logits)).item()
is_close = torch.allclose(our_logits, hf_logits, atol=1e-4)

print("-" * 55)
print(f"Maksimum Mutlak Fark (Max Absolute Diff): {max_abs_diff:.8e}")
print(f"Ortalama Mutlak Fark (Mean Absolute Diff): {mean_abs_diff:.8e}")
print(f"Fark tolerans dahilinde mi (atol=1e-4)  : {is_close}")
print("-" * 55)


Hugging Face 'gpt2' modeli yükleniyor...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Tüm ağırlıklar başarıyla eşlendi ve kopyalandı!


Test Girdisi: 'Hello, I'm a language model,'
Girdi Token ID'leri: [15496, 11, 314, 1101, 257, 3303, 2746, 11]
Girdi Tensör Boyutu: torch.Size([1, 8])


Bizim Model Logits Boyutu : torch.Size([1, 8, 50257])
HF Model Logits Boyutu    : torch.Size([1, 8, 50257])


-------------------------------------------------------
Maksimum Mutlak Fark (Max Absolute Diff): 6.10351562e-05
Ortalama Mutlak Fark (Mean Absolute Diff): 1.50337528e-05
Fark tolerans dahilinde mi (atol=1e-4)  : True
-------------------------------------------------------


### 5. Sıradaki Token Tahminlerinin (Top-5) Birebir Karşılaştırılması
Son token için her iki modelin ürettiği ilk 5 en olası adayı ve olasılıklarını karşılaştırıyoruz.

In [5]:
# Son pozisyondaki logitler
our_last_logits = our_logits[0, -1, :]
hf_last_logits = hf_logits[0, -1, :]

our_probs = F.softmax(our_last_logits, dim=-1)
hf_probs = F.softmax(hf_last_logits, dim=-1)

our_top5_probs, our_top5_indices = torch.topk(our_probs, 5)
hf_top5_probs, hf_top5_indices = torch.topk(hf_probs, 5)

print(f"{'Sıra':<5} | {'Bizim Model Token':<20} {'Olasılık':<12} | {'HF Model Token':<20} {'Olasılık':<12}")
print("-" * 75)
for i in range(5):
    our_t = enc.decode([our_top5_indices[i].item()])
    hf_t = enc.decode([hf_top5_indices[i].item()])
    print(f"{i+1:<5} | {repr(our_t):<20} {our_top5_probs[i].item():.4%}      | {repr(hf_t):<20} {hf_top5_probs[i].item():.4%}")


Sıra  | Bizim Model Token    Olasılık     | HF Model Token       Olasılık    
---------------------------------------------------------------------------
1     | ' not'               16.4341%      | ' not'               16.4341%
2     | ' and'               15.1580%      | ' and'               15.1578%
3     | ' I'                 10.7493%      | ' I'                 10.7494%
4     | ' so'                7.4222%      | ' so'                7.4222%
5     | ' but'               7.2490%      | ' but'               7.2490%


### 6. Hafta 8'deki GPT'den Farkları ve Transpoze Nedeni (Detaylı Analiz)

#### A) Hafta 8'deki Modelimizden Farklar (Madde Madde)

1. **Aktivasyon Fonksiyonu (GELU vs ReLU):**
   - **Hafta 8:** MLP katmanında standart `nn.ReLU()` kullanılmıştı. ReLU, 0'ın altındaki tüm negatif girdileri sıfırlar ve bu durum derin ağlarda bazı nöronların tamamen etkisiz hale gelmesine ("dying ReLU" problemi) yol açabilir.
   - **GPT-2:** Gaussian Error Linear Unit olan `nn.GELU(approximate='tanh')` kullanılır. GELU, negatif değerlerde yumuşak ve türevlenebilir bir eğriye sahiptir; bu sayede gradyanlar akmaya devam eder ve model çok daha zengin doğrusal olmayan temsiller öğrenebilir.

2. **Q, K, V Projeksiyonunun Tek Katmanda Birleşik Hesaplanması:**
   - **Hafta 8:** Query, Key ve Value için ayrı ayrı 3 tane lineer katman (`self.key`, `self.query`, `self.value`) ve her kafa için ayrı modüller kullanılmıştı.
   - **GPT-2:** Q, K ve V vektörleri tek bir büyük katmanda (`c_attn = nn.Linear(n_embd, 3 * n_embd)`) tek bir matris çarpımıyla hesaplanır ve ardından `.split()` ile ayrılır. GPU'da 3 ayrı küçük çarpım yapmak yerine tek bir büyük matris çarpımı yapmak bellek transfer gecikmesini (memory latency) azaltır ve hesaplama verimini katbekat artırır.

3. **Pre-LayerNorm Mimarisi ve Final LayerNorm (`ln_f`):**
   - **Hafta 8:** Klasik mimaride LayerNorm genellikle alt katmanların veya residual bağlantıların çıkışına uygulanırdı (Post-LN).
   - **GPT-2:** Normalizasyon blokların hemen girişine yerleştirilmiştir (`x = x + attn(ln_1(x))` ve `x = x + mlp(ln_2(x))`). Bu yapı (Pre-LN), residual akış yolunu ("residual highway") temiz tutar; böylece gradyanlar onlarca katman geriye doğru hiçbir zayıflama veya patlama olmadan doğrudan akabilir. Ayrıca blokların en sonunda, kelime projeksiyonu öncesi ek bir `ln_f` (Final LayerNorm) yer alır.

4. **Ağırlık Paylaşımı / Weight Tying (`wte` ve `lm_head`):**
   - **Hafta 8:** Giriş kelime gömme matrisi (`token_embedding_table`) ile çıkıştaki `lm_head` birbirinden tamamen bağımsız iki ayrı ağırlık kümesiydi.
   - **GPT-2:** Giriş gömme matrisi ile çıkış sınıflandırma matrisi aynı tensörü paylaşır (`self.transformer.wte.weight = self.lm_head.weight`). Bu sayede 50,257 x 768 = **38,597,376 adet parametre tasarrufu** sağlanır ve modelin girdi/çıktı anlamsal uzayı birbirine bağlanarak düzenlileştirme (regularization) etkisi elde edilir.

5. **Kapasite ve Tokenizer Boyutu:**
   - **Hafta 8:** Küçük toy model (blok boyutu 256, gömme boyutu 384, 6 katman) ve karakter seviyesinde küçük bir sözlük (94 karakter).
   - **GPT-2:** 1024 blok boyutu, 768 gömme boyutu, 12 katman, 12 kafa ve 50,257 boyutlu BPE (Byte-Pair Encoding) alt-kelime sözlüğü.

---

#### B) Bazı Ağırlıkları Yüklerken Neden Transpoze Etmemiz Gerekiyor?

Hugging Face'ten GPT-2 ağırlıklarını yüklerken şu katmanların ağırlık matrislerinin transpozesini (`.t()`) almak zorunda kaldık:
- `attn.c_attn.weight`
- `attn.c_proj.weight`
- `mlp.c_fc.weight`
- `mlp.c_proj.weight`

**Nedeni:**
1. **Orijinal TensorFlow / OpenAI Tasarımı:** OpenAI ekibi GPT-2 modelini TensorFlow 1.x kullanarak geliştirmişti. Bu modelde lineer katmanlar standart yoğun katman (`Dense`) yerine özel bir 1-boyutlu evrişim katmanı olan `tf.layers.conv1d` (veya `Conv1D`) şeklinde yazılmıştı.
2. **Matris Boyutlandırma Farkı:**
   - TensorFlow `Conv1D` modülü ağırlık matrisini `[giriş_boyutu, çıkış_boyutu]` (`[in_features, out_features]`) şeklinde saklar.
   - Buna karşın standart PyTorch `nn.Linear(in_features, out_features)` sınıfı ağırlık tensörünü matematiksel formülasyonu ($y = x A^T + b$) gereği `[çıkış_boyutu, giriş_boyutu]` (`[out_features, in_features]`) şeklinde bekler.
3. Hugging Face, OpenAI'ın orijinal ağırlıklarını saf haliyle koruduğu için bu tensörlerin boyutları `(768, 2304)`, `(768, 768)` ve `(768, 3072)` biçimindedir. Eğer transpoze etmeden PyTorch katmanlarına atamaya kalkarsak boyut uyuşmazlığı hatası alırız ya da çarpım yönü ters olacağı için anlamsız logits elde ederiz.
4. Bu yüzden ilgili tüm `Conv1D` katmanlarını PyTorch'un standart `nn.Linear` katmanına aktarırken `.t()` ile matrisin devriğini (transpozesini) almak şarttır.
